In [2]:
%pip install fashion-clip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.3 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=0ca20ef40da609cbd509bfdcb2309a84334657b607007e9d4f728010868f23db
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy


In [8]:
%pip install --upgrade --force-reinstall "transformers==4.48.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 462.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
import os
import json
import pickle
import requests
import numpy as np
import pandas as pd
import zipfile
from io import BytesIO
from PIL import Image
from tqdm.auto import tqdm
from fashion_clip.fashion_clip import FashionCLIP

In [9]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os

own_dir = "/content/drive/MyDrive/Colab Notebooks/AIS-IRS/practice_module/stage1_student4/polyvore"
os.chdir(own_dir)

## product images embedding

In [11]:
zip_path = "polyvore_images.zip"
z = zipfile.ZipFile(zip_path, "r")

image_paths = [name for name in z.namelist() if name.lower().endswith(".jpg")]
print("Total images:", len(image_paths))

Total images: 8850


In [12]:
model = FashionCLIP("fashion-clip")

In [13]:
batch_size = 32

embedding_dict = {}

with zipfile.ZipFile(zip_path, "r") as z:

    image_paths = [
        name for name in z.namelist()
        if name.startswith("selected_images/") and name.lower().endswith(".jpg")
    ]
    print("Total images:", len(image_paths))

    # Generate embeddings
    for i in tqdm(range(0, len(image_paths), batch_size)):

        batch_paths = image_paths[i:i + batch_size]
        batch_images = []
        batch_ids = []

        for path in batch_paths:
            try:
                image = Image.open(BytesIO(z.read(path))).convert("RGB")
                product_id = path.split("/")[-1].removesuffix(".jpg")
                batch_images.append(image)
                batch_ids.append(product_id)

            except Exception as e:
                print(f"Failed: {path}, {e}")

        if not batch_images:
            continue

        batch_embeddings = model.encode_images(images=batch_images, batch_size=len(batch_images))

        for product_id, embedding in zip(batch_ids, batch_embeddings):
            embedding_dict[product_id] = embedding

# Save embeddings
np.savez("fashionclip_embeddings.npz", **embedding_dict)

print("Total embedded products:", len(embedding_dict))
print("Embedding shape:", next(iter(embedding_dict.values())).shape)

Total images: 8850


  0%|          | 0/277 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:03<00:00,  3.02s/it]


Total embedded products: 8850
Embedding shape: (512,)
